# Phase 24 — Cross-national transfer: why are the external AUCs ≈ 0.48-0.49?
**Resolves:** Reviewer 1 point 3 (AUC < 0.5 must not be called "limited transfer") and R2-5-style direction/transport concerns for Study 2;
supports an accurate, conservative description in the Abstract, Results, Discussion and Conclusion.

Uses ONLY the saved harmonised files (`pakistan_harmonized.csv`, `arab5_harmonized.csv`) — no tuning, no retraining of the Phase 13 forests.
 (1) Odds ratio of each of the five harmonised features with the PHQ-9 depression label in each cohort (is the DIRECTION the same?);
 (2) single-feature transfer: score = source-cohort direction of each feature, AUC in the target;
 (3) a transparent 5-feature logistic regression trained in one cohort and applied frozen to the other (shows the failure is not a random-forest artefact);
     bootstrap CI and two-sided bootstrap p-value against AUC = 0.5;
 (4) profile-level agreement: observed depression prevalence per harmonised profile (32 possible) in both cohorts (Spearman).
**Outputs:** `S2_xnat_feature_odds_ratios.csv`, `S2_xnat_single_feature_transfer.csv`, `S2_xnat_logistic_transfer.csv`, `S2_xnat_profile_agreement.csv`

In [1]:
import os, sys, json, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..", "src")))
from revision_utils import *

ENV = get_env()          # paths + config (notebooks must be run from the notebooks/ folder)
SEED = ENV["SEED"]       # 42
B = ENV["N_BOOT"]        # 2000 bootstrap resamples (config.yaml)
print("Project root:", ENV["ROOT"], "| outputs ->", ENV["REV"])

Project root: C:\maternal-mental-health-risk | outputs -> C:\maternal-mental-health-risk\reports/tables\revision


In [2]:
from harmonization import HARMONIZED_FEATURE_COLS as HF
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
pk = pd.read_csv(os.path.join(ENV["EXTERNAL"], "pakistan_harmonized.csv")).dropna(subset=HF + ["y_depression"]).reset_index(drop=True)
ar = pd.read_csv(os.path.join(ENV["EXTERNAL"], "arab5_harmonized.csv")).dropna(subset=HF + ["y_depression"]).reset_index(drop=True)
COH = {"Pakistan": pk, "Arab-region": ar}
RISK = {"age_group": "35_plus", "education_group": "secondary_or_less", "employed": "employed",
        "financial_strain": "strain", "first_pregnancy": "first_pregnancy"}
print({k: (len(v), round(v.y_depression.mean(), 3)) for k, v in COH.items()})

or_rows = []
for cn, df in COH.items():
    y = df.y_depression.astype(int)
    for f in HF:
        ex = df[f] == RISK[f]
        a, b, c, d_ = int((ex & (y == 1)).sum()), int((ex & (y == 0)).sum()), int((~ex & (y == 1)).sum()), int((~ex & (y == 0)).sum())
        o, lo, hi = odds_ratio_ci(a, b, c, d_)
        or_rows.append(dict(cohort=cn, feature=f, exposed_level=RISK[f], n_exposed=int(ex.sum()), prev_exposed=a / max(a + b, 1),
                            prev_unexposed=c / max(c + d_, 1), odds_ratio=o, or_ci_lower=lo, or_ci_upper=hi))
orr = pd.DataFrame(or_rows)
piv = orr.pivot(index="feature", columns="cohort", values="odds_ratio")
orr = orr.merge(pd.DataFrame({"feature": piv.index, "direction_agrees_across_cohorts": ((piv["Pakistan"] - 1) * (piv["Arab-region"] - 1) > 0).values}), on="feature")
save_table(orr, ENV, "S2_xnat_feature_odds_ratios.csv"); print(orr.round(3).to_string())

{'Pakistan': (14002, np.float64(0.648)), 'Arab-region': (1498, np.float64(0.457))}
Saved: C:\maternal-mental-health-risk\reports/tables\revision\S2_xnat_feature_odds_ratios.csv  (10 rows)
        cohort           feature      exposed_level  n_exposed  prev_exposed  prev_unexposed  odds_ratio  or_ci_lower  or_ci_upper  direction_agrees_across_cohorts
0     Pakistan         age_group            35_plus       1856         0.670           0.645       1.120        1.010        1.243                            False
1     Pakistan   education_group  secondary_or_less      10049         0.653           0.636       1.075        0.995        1.160                             True
2     Pakistan          employed           employed       3640         0.627           0.655       0.885        0.818        0.957                             True
3     Pakistan  financial_strain             strain       7140         0.639           0.657       0.926        0.864        0.993                          

In [3]:
def boot_auc_p(y, s, B=B, seed=SEED):
    rng = np.random.RandomState(seed); y = np.asarray(y); s = np.asarray(s); v = []
    for _ in range(B):
        i = rng.randint(0, len(y), len(y))
        if len(np.unique(y[i])) == 2: v.append(roc_auc_score(y[i], s[i]))
    v = np.array(v); return np.percentile(v, 2.5), np.percentile(v, 97.5), min(1.0, 2 * min(np.mean(v <= 0.5), np.mean(v >= 0.5)))
from sklearn.metrics import roc_auc_score
# (2) single-feature transfer with the SOURCE direction
sf = []
for src, tgt in [("Pakistan", "Arab-region"), ("Arab-region", "Pakistan")]:
    for f in HF:
        o_src = float(orr[(orr.cohort == src) & (orr.feature == f)].odds_ratio.iloc[0]); sign = 1.0 if o_src > 1 else -1.0
        ind = (COH[tgt][f] == RISK[f]).astype(float).values * sign
        yt = COH[tgt].y_depression.astype(int).values
        lo, hi, p = boot_auc_p(yt, ind)
        sf.append(dict(source=src, target=tgt, feature=f, source_direction="higher risk if exposed" if sign > 0 else "lower risk if exposed",
                       target_auc=roc_auc_score(yt, ind), ci_lower=lo, ci_upper=hi, p_vs_0_5=p))
sf = pd.DataFrame(sf); save_table(sf, ENV, "S2_xnat_single_feature_transfer.csv"); print(sf.round(3).to_string())

# (3) transparent logistic regression, frozen transfer
def design(df): return pd.get_dummies(df[HF], drop_first=False).astype(float)
lg = []
for src, tgt in [("Pakistan", "Arab-region"), ("Arab-region", "Pakistan")]:
    Xs, Xt = design(COH[src]), design(COH[tgt]).reindex(columns=design(COH[src]).columns, fill_value=0.0)
    ys, yt = COH[src].y_depression.astype(int).values, COH[tgt].y_depression.astype(int).values
    m = LogisticRegression(C=1.0, max_iter=2000).fit(Xs, ys)
    cv = cross_val_predict(LogisticRegression(C=1.0, max_iter=2000), Xs, ys, cv=StratifiedKFold(5, shuffle=True, random_state=SEED), method="predict_proba")[:, 1]
    ext = m.predict_proba(Xt)[:, 1]; lo, hi, p = boot_auc_p(yt, ext)
    lg.append(dict(source=src, target=tgt, internal_cv_auc_ordinary=roc_auc_score(ys, cv), external_auc=roc_auc_score(yt, ext),
                   ci_lower=lo, ci_upper=hi, p_vs_0_5=p, n_source=len(ys), n_target=len(yt),
                   ci_includes_0_5=bool(lo <= 0.5 <= hi)))
lg = pd.DataFrame(lg); save_table(lg, ENV, "S2_xnat_logistic_transfer.csv"); print(lg.round(3).to_string())

# (4) profile-level agreement
from scipy.stats import spearmanr
def prof(df, nm):
    key = df[HF].astype(str).agg("|".join, axis=1); g = df.assign(profile=key).groupby("profile").y_depression.agg(["mean", "size"]); g.columns = [f"prev_{nm}", f"n_{nm}"]; return g
pp = prof(pk, "pakistan").join(prof(ar, "arab5"), how="inner")
pp = pp[(pp.n_pakistan >= 10) & (pp.n_arab5 >= 10)]
rho, prho = spearmanr(pp.prev_pakistan, pp.prev_arab5) if len(pp) >= 3 else (np.nan, np.nan)
pp["spearman_rho_all_profiles"] = rho; pp["spearman_p"] = prho
save_table(pp.reset_index(), ENV, "S2_xnat_profile_agreement.csv"); print(f"profiles with n>=10 in both cohorts: {len(pp)}; Spearman rho = {rho:.3f} (p = {prho:.3f})")

Saved: C:\maternal-mental-health-risk\reports/tables\revision\S2_xnat_single_feature_transfer.csv  (10 rows)
        source       target           feature        source_direction  target_auc  ci_lower  ci_upper  p_vs_0_5
0     Pakistan  Arab-region         age_group  higher risk if exposed       0.496     0.479     0.515     0.687
1     Pakistan  Arab-region   education_group  higher risk if exposed       0.525     0.504     0.545     0.017
2     Pakistan  Arab-region          employed   lower risk if exposed       0.538     0.514     0.562     0.006
3     Pakistan  Arab-region  financial_strain   lower risk if exposed       0.435     0.410     0.459     0.000
4     Pakistan  Arab-region   first_pregnancy   lower risk if exposed       0.513     0.488     0.538     0.269
5  Arab-region     Pakistan         age_group   lower risk if exposed       0.494     0.488     0.499     0.030
6  Arab-region     Pakistan   education_group  higher risk if exposed       0.507     0.500     0.515     0

**Wording guide (Reviewer 1, point 3).** State both external AUCs with their CIs and say what they show:
* Pakistan -> Arab-region: AUC 0.480 (0.451-0.510): CI includes 0.5 -> "indistinguishable from chance".
* Arab-region -> Pakistan: AUC 0.488 (0.478-0.498): CI excludes 0.5 narrowly -> "marginally below chance", i.e. no transferable ranking,
  and for the harmonised five-feature model the ranking is weakly inverted in the target cohort.
Replace "limited raw predictive transfer" with "no useful transfer (AUC at or marginally below 0.5)". Use this phase's odds-ratio table to
explain WHY (opposite or non-comparable directions for features such as financial strain/employment between the two instruments), and keep the
claim about "consistent SHAP rankings" to what it is: the same variables were ranked highest, not that their effects agree in sign.